In [1]:
from pathlib import Path
from urllib.request import urlopen

REPO = "contrivancecompanychicago/imhotep-bizarro-secretary-backend"
COMMIT = "0630117f4228f82ba39c2ae4ea5dacd9dcaadba0"

research_dir = Path("/content/imhotep-research")
research_dir.mkdir(parents=True, exist_ok=True)

for filename in ["README.md", "baseline.py"]:
    url = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/research/{filename}"
    with urlopen(url, timeout=30) as response:
        (research_dir / filename).write_bytes(response.read())
    print(f"Loaded: {filename}")

print("\nSETUP COMPLETE")
print("Repository commit:", COMMIT)

Loaded: README.md
Loaded: baseline.py

SETUP COMPLETE
Repository commit: 0630117f4228f82ba39c2ae4ea5dacd9dcaadba0


In [2]:
%run /content/imhotep-research/baseline.py

{
  "label": "synthetic smoke demo only; not a study result",
  "method": "token_overlap_v1",
  "query_count": 2,
  "top1_accuracy": 1.0,
  "rankings": [
    {
      "query_id": "Q1",
      "ranked_ids": [
        "SYN-001-N1",
        "SYN-002-N1"
      ],
      "relevant_id": "SYN-001-N1"
    },
    {
      "query_id": "Q2",
      "ranked_ids": [
        "SYN-003-N1"
      ],
      "relevant_id": "SYN-003-N1"
    }
  ]
}


In [3]:
import runpy

baseline = runpy.run_path("/content/imhotep-research/baseline.py")
rank = baseline["rank"]

documents = [
    {
        "id": "SYN-001-N1",
        "text": "Fictional case SYN-001 requests an afternoon appointment."
    },
    {
        "id": "SYN-002-N1",
        "text": "Fictional case SYN-002 requests a morning appointment."
    }
]

query = "visit after noon"

print("Query:", query)
print("Expected document: SYN-001-N1")
print("Retrieved documents:", rank(query, documents))

Query: visit after noon
Expected document: SYN-001-N1
Retrieved documents: []


In [4]:
queries = [
    ("Exact wording", "afternoon appointment"),
    ("Different wording", "visit after noon"),
]

expected = "SYN-001-N1"

for label, query in queries:
    retrieved = rank(query, documents)
    correct = bool(retrieved) and retrieved[0] == expected

    print(f"{label}: {query}")
    print(f"Retrieved: {retrieved}")
    print(f"Correct first result: {correct}\n")

Exact wording: afternoon appointment
Retrieved: ['SYN-001-N1', 'SYN-002-N1']
Correct first result: True

Different wording: visit after noon
Retrieved: []
Correct first result: False



In [6]:
"""
### Development observation 001: Query wording

Method: token_overlap_v1

Documents: Two fictional appointment records, held fixed.

Results:
- "afternoon appointment" retrieved the correct document first.
- "visit after noon" returned no documents.
- Top-1 accuracy across these two development queries: 1/2 = 50%.

Interpretation:
The token-overlap baseline missed a relevant document when the query
used different words with a similar intended meaning.

Limitation:
These are two hand-authored development examples, not a held-out
evaluation or evidence of clinical effectiveness.

Next experiment:
Compare the unchanged baseline with a clearly labeled query-expansion
method on these development examples before testing unseen cases.
"""

'\n### Development observation 001: Query wording\n\nMethod: token_overlap_v1\n\nDocuments: Two fictional appointment records, held fixed.\n\nResults:\n- "afternoon appointment" retrieved the correct document first.\n- "visit after noon" returned no documents.\n- Top-1 accuracy across these two development queries: 1/2 = 50%.\n\nInterpretation:\nThe token-overlap baseline missed a relevant document when the query\nused different words with a similar intended meaning.\n\nLimitation:\nThese are two hand-authored development examples, not a held-out\nevaluation or evidence of clinical effectiveness.\n\nNext experiment:\nCompare the unchanged baseline with a clearly labeled query-expansion\nmethod on these development examples before testing unseen cases.\n'

In [7]:
import re

def expand_query(query):
    expanded = query.lower()
    expanded = re.sub(r"\bafter noon\b", "afternoon", expanded)
    expanded = re.sub(r"\bvisit\b", "appointment", expanded)
    return expanded

methods = {
    "Original baseline": lambda query: query,
    "Rule-based expansion v1": expand_query,
}

for method_name, transform in methods.items():
    correct_count = 0
    print(f"\n{method_name}")

    for label, query in queries:
        search_query = transform(query)
        retrieved = rank(search_query, documents)
        correct = bool(retrieved) and retrieved[0] == expected
        correct_count += int(correct)

        print(f"  Original: {query}")
        print(f"  Searched: {search_query}")
        print(f"  Correct first result: {correct}")

    print(f"  Top-1 accuracy: {correct_count}/{len(queries)}")


Original baseline
  Original: afternoon appointment
  Searched: afternoon appointment
  Correct first result: True
  Original: visit after noon
  Searched: visit after noon
  Correct first result: False
  Top-1 accuracy: 1/2

Rule-based expansion v1
  Original: afternoon appointment
  Searched: afternoon appointment
  Correct first result: True
  Original: visit after noon
  Searched: appointment afternoon
  Correct first result: True
  Top-1 accuracy: 2/2


In [8]:
# Neither document describes an appointment before noon.
negative_query = "visit before noon"

for method_name, transform in methods.items():
    searched = transform(negative_query)
    retrieved = rank(searched, documents)

    print(f"\n{method_name}")
    print("Query:", negative_query)
    print("Searched:", searched)
    print("Retrieved:", retrieved)


Original baseline
Query: visit before noon
Searched: visit before noon
Retrieved: []

Rule-based expansion v1
Query: visit before noon
Searched: appointment before noon
Retrieved: ['SYN-001-N1', 'SYN-002-N1']


In [9]:
missing_query = "evening appointment"

for method_name, transform in methods.items():
    retrieved = rank(transform(missing_query), documents)
    print(f"{method_name}: {retrieved}")

Original baseline: ['SYN-001-N1', 'SYN-002-N1']
Rule-based expansion v1: ['SYN-001-N1', 'SYN-002-N1']


In [10]:
def rank_with_time_filter(query, documents):
    expanded = expand_query(query)
    expanded = re.sub(r"\bbefore noon\b", "morning", expanded)

    time_terms = {"morning", "afternoon", "evening"}
    required_times = set(expanded.split()) & time_terms

    eligible = [
        doc for doc in documents
        if required_times.issubset(set(doc["text"].lower().split()))
    ]
    return rank(expanded, eligible)

for query in [
    "afternoon appointment",
    "visit after noon",
    "visit before noon",
    "evening appointment",
]:
    print(query, "→", rank_with_time_filter(query, documents))

afternoon appointment → ['SYN-001-N1']
visit after noon → ['SYN-001-N1']
visit before noon → ['SYN-002-N1']
evening appointment → []
